# 2023_01_04

## using optuna

데이터셋이 다음과 같이 세 가지로 만들어 놓음
* 사건 발생 count 예측
* 사건 발생 count 에다가 spital aggeration 적용후 예축
* target을 사건발생 count와 spital aggeration count를 동시에 예측 

optuna를 이용해서 최적의 결과값을 찾기  

> optuna tuning notebook : http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/2022_12_28_TEST_new_data_training_spatial_aggregate/optuna.ipynb

## reference model
TFT model로 어느정도 튜닝이 끝남  
더 이상 TFT를 이용해서 성능을 끌어올리기에는 무리가 있어보임  



다음과 같은 이유에서 reference model을 만들어서 성능 평가가 필요함  

* 논문을 쓰기 위해서는 Reference model로 어느 정도 성능이 좋아졌는지를 보일 필요가 있음  
* Ensemble을 통한 성능 올리기

reference model로 다음과 같은 모델을 사용함
* XGBoost
* MultivariateTSF LSTM

기존의 모델을 참고해서 학습 및 성능 평가  

#### check point
* emwa 적용 필요성 
* keras에서 f2 score를 metics으로 사용하는 방법
* test data split 여부
* ref model을은 단일 구역에 대한 예측을 하도록 만들어짐 이를 춘천시 각 행정동마다 학습 및 평가 진행
* TFT 모델에서 encoder_length만큼의 길이를 읽어서 prediction length만큼 예측을 하는데 이를 ref 모델에서는 어떻게 설정  
* pytorch_forecasting에서 제공해주는 전처리들(각 feautre의 normailzation과 같은 기능들을 직접 처리해야 함)


## decision_tree_base_model

기존의 코드를 참조 한 했을 때 다음과 같은 특징들이 있음

* EDA 확인
* feature engineering
* up sampling, down sampling

XGBoost의 classification과 regression 버전 둘다 만듬

> result notebook : http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/reference%20model/decision_tree_base_model.ipynb#

####  EDA
각 행정동별로 col별로 상관 관계 확인 
![](https://i.imgur.com/G8I90Zy.png)


####  feauture engineering

count 값을 제외한 모든 변수들에 대해서 MinMaxScaler를 적용  
```python
data_features = ['pops', 'windspd','humid', 'temp', 'precip_form', 'precip', 'isHoliday']
data = pd.read_csv('../data/2016_2021.csv')
data['isHoliday'] = data['isHoliday'].astype(int)
data['precip_form'] = data['precip_form'].astype(int)
data['count'] = data['count'].astype(float)


# feauter scaing
minmax_scaler = MinMaxScaler()
for col in data.columns[4:]:
    X = data[col].values.reshape(-1,1)
    data[col] = minmax_scaler.fit_transform(X)
```

#### optuna training

```python
import optuna

def training_xgboost_regressor(objective , n_estimators,learning_rate , max_depth ):
    result_list = []
    for idx, dong in enumerate(data['h_dong'].unique()):
        dong_data = data[data['h_dong']==dong]
        features = dong_data[data_features]
        target = dong_data['count']
        X_train, X_test, y_train, y_test = train_test_split(features, target , test_size = 0.03)
        regressor = XGBRegressor(objective, n_estimators,learning_rate  , max_depth, use_label_encoder=False )
        regressor.fit(X_train,y_train)
        pred = regressor.predict(X_test)
        pred = pred.round()
        result_list.append(metrics.fbeta_score(y_test,pred,2,average = 'weighted'))
    return np.mean(result_list)
```
* label : 'count'
* target :  ['pops', 'windspd','humid', 'temp', 'precip_form', 'precip', 'isHoliday']  
으로 학습 진행  
* f2 score를 optuna의 objective로 사용 


```python
def regressor_confusion_matrix(regressor):
    ret_df = pd.DataFrame()
    for idx, dong in enumerate(data['h_dong'].unique()):
        dong_data = data[data['h_dong']==dong]
        features = dong_data[data_features]
        target = dong_data['count']
        X_train, X_test, y_train, y_test = train_test_split(features, target , test_size = 0.03)
        regressor.fit(X_train,y_train)
        
        pred = regressor.predict(X_test)
        pred = pred.round()
        conf_mat =  metrics.confusion_matrix(y_test,pred)
        fig = px.imshow(conf_mat,labels =dict(x = "model" , y = 'real'), text_auto = True ,title =  f'{dong} confusion matrix')
        fig.show()
```
* 각 동별로 confusion matrix 확인

``` python

def objective(trail):
    param = {
        'objective' : trail.suggest_categorical('objective',
        ['reg:squarederror', 'reg:squaredlogerror','reg:absoluteerror' , 'reg:pseudohubererror','count:poisson' , 'aft_loss_distribution'
        'multi:softmax' , 'rank:ndcg']),
        'n_estimators' : trail.suggest_categorical('n_estimators',[4,8,12,16, 20,]),
        'learning_rate' : trail.suggest_categorical('learning_rate',[0.01 ,0.05, 0.1 ,0.3]),
        'max_depth' : trail.suggest_categorical('max_depth' , [2, 4, 6, 8, 10])
    }
    
    return training_xgboost_regressor(**param)
```

* optuna를 이용한 최적화 
* https://xgboost.readthedocs.io/en/stable/parameter.html
* 위 사이트에서 objective funs과 같은 params 설정 가능  


#### metric 

metric이 현재 데이터에 적합하지 않음

* 각 동별 metric

|  | h_dong | f1_score | f2_score | recall | precision |
|---:|---:|---:|---:|---:|---:|
| 0 | 석사동 | 0.954852 | 0.963005 | 0.968519 | 0.941565 |
| 1 | 신동면 | 0.979367 | 0.983218 | 0.985802 | 0.973016 |
| 2 | 강남동 | 0.963109 | 0.969658 | 0.974074 | 0.952388 |
| 3 | 서 면 | 0.971993 | 0.976561 | 0.979630 | 0.964474 |
| 4 | 남 면 | 0.987052 | 0.988893 | 0.990123 | 0.984000 |
| 5 | 사북면 | 0.986129 | 0.988153 | 0.989506 | 0.982775 |
| 6 | 동내면 | 0.958818 | 0.965716 | 0.970370 | 0.947538 |
| 7 | 효자1동 | 0.961572 | 0.967934 | 0.972222 | 0.951152 |
| 8 | 남산면 | 0.953304 | 0.960544 | 0.965432 | 0.941476 |
| 9 | 근화동 | 0.979980 | 0.983465 | 0.985802 | 0.974226 |
| 10 | 신사우동 | 0.971080 | 0.976559 | 0.980247 | 0.962083 |
| 11 | 신북읍 | 0.968932 | 0.974588 | 0.978395 | 0.959650 |
| 12 | 소양동 | 0.978137 | 0.981985 | 0.984568 | 0.971789 |
| 13 | 약사명동 | 0.987357 | 0.988645 | 0.989506 | 0.985217 |
| 14 | 동 면 | 0.956067 | 0.963499 | 0.968519 | 0.943931 |
| 15 | 후평1동 | 0.960963 | 0.967687 | 0.972222 | 0.949962 |
| 16 | 퇴계동 | 0.960654 | 0.967195 | 0.971605 | 0.949947 |
| 17 | 북산면 | 0.986129 | 0.988153 | 0.989506 | 0.982775 |
| 18 | 조운동 | 0.987056 | 0.989633 | 0.991358 | 0.982791 |
| 19 | 동산면 | 0.979362 | 0.982479 | 0.984568 | 0.974210 |
| 20 | 교 동 | 0.996605 | 0.996790 | 0.996914 | 0.996297 |


* 강남동 confusion matrix

![](https://i.imgur.com/YnSaaZr.png)

* 사건을 예측 못하고 있는데도 TN이 많기 때문에 TN을 높이는 쪽으로 학습   
* `(metrics.fbeta_score(y_test,pred,2,average = 'weighted')` 
* f2 score를 계산할 때 average = 'weighted' 옵션을 넣어줘도 그럼  